<a href="https://colab.research.google.com/github/PrajwalArali/QAmodel/blob/main/QAmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# FLAN-T5 (for classification tasks)
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

# Falcon-7B-Instruct (for generation tasks)
falcon_tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-7b-instruct")
falcon_model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b-instruct",
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

In [ ]:
document_title = "The Goat"
document_text = (
    "Virat Kohli (born 5 November 1988) is an Indian international cricketer who plays ODI cricket for the national team and is a former captain in all formats. He is a right-handed batsman and occasional right-arm medium pace bowler. Considered one of the greatest all-format batsmen in the history of cricket, he is called the King, the Chase Master, and the Run Machine for his skills, records and ability to lead his team to victory. Kohli is the highest run-scorer in the Indian Premier League, third in T20I, third in ODI, and third in international cricket. He has the most ODI centuries and second-most centuries in international cricket. Kohli is also the most successful Test captain of India with back-to-back Test mace wins and most victories in his tenure. He is the only batter to earn 900 rating points in all three formats."
"Kohli was the captain of the 2008 U19 World Cup winning team and was a crucial member of the teams that won 2011 ODI World Cup, 2013 Champions Trophy, 2024 T20 World Cup, and 2025 Champions Trophy. He plays for Royal Challengers Bengaluru in the Indian Premier League and for Delhi in domestic cricket. In 2013, Kohli was ranked number one in the ODI batting rankings. In 2015, he achieved the same in T20I. In 2018, he was ranked number one in Test, making him the only Indian to hold the number one spot in all three formats. He is the first player to score 20,000 runs in a decade. He was the Cricketer of the Decade for 2011 to 2020."
"Kohli has won ten ICC Awards, making him the most awarded player in international cricket history. He won the ODI Player of the Year award four times in 2012, 2017, 2018, and 2023. He won the Cricketer of the Year award, on two occasions, in 2017 and 2018. In 2018, he became the first player to win all three major awards including Cricketer of the Year, ODI Player of the Year and Test Player of the Year in the same year. He was honored with the Spirit of Cricket Award in 2019 and given the Cricketer of the Decade and ODI Cricketer of the Decade in 2020. Kohli was named the Wisden Leading Cricketer in the World for three consecutive years."
"Kohli has the most Player of the Series and second most Player of the Match awards to his name in all three formats combined. He was honoured with the Arjuna Award in 2013, the Padma Shri in 2017, and India's highest sporting honour, the Khel Ratna Award, in 2018. Time included him on its 100 most influential people in the world list in 2018."
"After winning the 2024 T20 World Cup and winning the Player of the Match award in the final, Kohli announced his retirement from T20Is. On 12 May 2025, aged 36, he announced his retirement from the Test format. He is married to actress Anushka Sharma, and they have two children."

"Early life"
"Kohli was born on 5 November 1988 in Delhi into a Punjabi Hindu family. His mother Saroj Kohli is as a housewife while his father Prem Nath Kohli worked as a criminal lawyer. He has an elder brother Vikas and an elder sister Bhawna. His formative years were spent in Uttam Nagar. His early education was at Vishal Bharti Public School.[15] As per his family, Kohli exhibited an early affinity for cricket as a 3-year-old. He would pick up a bat and request his father bowl to him."
"In 1998, the West Delhi Cricket Academy was created. In May, his father arranged for him to meet Rajkumar Sharma. Upon the suggestion of their neighbours, Kohli's father considered enrolling his son in a professional cricket academy, as they believed his ability merited more than gully cricket. He was unable to secure a place in the U-14 Delhi team, due to extraneous factors. His father reportedly received offers to relocate his son to influential clubs, which would ensure his selection, but he declined the proposals."
"Kohli found his way into the U-15 team. He received training at the academy and participated in matches at the Sumeet Dogra Academy located at Vasundhara Enclave. In pursuit of furthering his cricketing career, he transferred to Saviour Convent School during his ninth-grade education."
"On 18 December 2006, his father died of a stroke. As per his mother, Kohli's demeanour shifted noticeably after his father's death. He took on cricket with newfound seriousness, prioritising playing time and dedicating himself fully to the sport. Kohli's family resided in Meera Bagh, Paschim Vihar until the year 2015, after which they relocated to Gurgaon."
)

document_sentences = [
    "S1: Virat Kohli (born 5 November 1988) is an Indian international cricketer",
    "S2: He plays ODI cricket for the national team and is a former captain in all formats.",
    "S3: He is the only batter to earn 900 rating points in all three formats.",
    "S4: He is a right-handed batsman and occasional right-arm medium pace bowler.",
    "S5: Considered one of the greatest all-format batsmen in the history of cricket, he is called the King, the Chase Master, and the Run Machine.",
    "S6: Kohli is the highest run-scorer in the Indian Premier League, third in T20I, third in ODI, and third in international cricket.",
    "S7: He has the most ODI centuries and second-most centuries in international cricket.",
    "S8: Kohli is also the most successful Test captain of India with back-to-back Test mace wins and most victories in his tenure.",
    "S9: Kohli was the captain of the 2008 U19 World Cup winning team and was a crucial member of the teams that won 2011 ODI World Cup, 2013 Champions Trophy, 2024 T20 World Cup, and 2025 Champions Trophy.",
    "S10: He was the Cricketer of the Decade for 2011 to 2020.",
    "S11: He is the first player to score 20,000 runs in a decade.",
    "S12: He plays for Royal Challengers Bengaluru in the Indian Premier League and for Delhi in domestic cricket.",
    "S13: In 2013, Kohli was ranked number one in the ODI batting rankings. In 2015, he achieved the same in T20I. In 2018, he was ranked number one in Test, making him the only Indian to hold the number one spot in all three formats."
]


In [ ]:
def generate_user_utterance(conversation):
    prompt = f"Document: {document_text}\nConversation:\n{conversation}\nUser:"
    inputs = falcon_tokenizer(prompt, return_tensors="pt").to(device)
    outputs = falcon_model.generate(
        **inputs, max_new_tokens=40, pad_token_id=falcon_tokenizer.eos_token_id
    )
    return falcon_tokenizer.decode(outputs[0], skip_special_tokens=True).split("User:")[-1].strip()



In [ ]:
def classify_answerability(question, conversation):
    prompt = f"""
You are given a document and a conversation. Determine if the last user question has an answer in the document.

Document:
{chr(10).join(document_sentences)}
EoD

Conversation:
{conversation}
User: {question}
Does the document contain an answer to the last user query? Yes/No:
"""
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = flan_model.generate(**inputs, max_new_tokens=10)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
def select_answer_sentences(question):
    prompt = f"""
Given a document and a user question, find sentence IDs that contain the answer.

Document:
{chr(10).join(document_sentences)}
EoD
Question: {question}
Answer sentences:
"""
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = flan_model.generate(**inputs, max_new_tokens=20)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


In [ ]:
def generate_agent_response(question, selected_sentences, conversation):
    prompt = f"Document: {document_text}\nUser: {question}\nAgent:"
    inputs = falcon_tokenizer(prompt, return_tensors="pt").to(device)
    outputs = falcon_model.generate(
        **inputs, max_new_tokens=60, pad_token_id=falcon_tokenizer.eos_token_id
    )
    return falcon_tokenizer.decode(outputs[0], skip_special_tokens=True).split("Agent:")[-1].strip()


In [ ]:
print("\n--- Starting QA Conversation Generation ---")
conversation = ""

for turn in range(5):
    user_question = input("User: ")
    if user_question.lower() in ["exit", "quit"]:
        print("Ending conversation.")
        break

    answerable = classify_answerability(user_question, conversation)
    if answerable.lower().startswith("yes"):
        relevant = select_answer_sentences(user_question)
        agent_response = generate_agent_response(user_question, relevant, conversation)
    else:
        agent_response = "Sorry, I can’t find an answer in the document."

    print(f"Agent: {agent_response}")
    conversation += f"User: {user_question}\nAgent: {agent_response}\n"